# Module 1.1: UniFi UNAS 2 Hardware Prep, Dual 4TB HDD Assembly & PoE++ Power Architecture

## UniFi Storage & Enterprise Networking (Chapter 3) - Step 1: Hardware Assembly & Physical Cabling

## 🎯 Learning Goals

By completing this laboratory module, you will be able to:

1. Analyze the hardware architecture of the **Ubiquiti UniFi NAS 2 (`UNAS-2`)** and **2× `UACC-HDD-S-4TB`** 3.5" SATA drives, including why **RAID 1** mirroring is mandatory over RAID 0 for homelab durability.
2. Explain the critical **PoE+ (`802.3at`, 32W) vs. PoE++ (`802.3bt`, 60W)** electrical incompatibility between the `USW-Enterprise-8-PoE` switch and the `UNAS 2`, and wire the **included 60W PoE++ injector** cleanly inline on your UPS.
3. Follow ESD-safe, powered-off drive installation procedures for the `UNAS 2`'s **non-hot-swappable** 3.5" SATA bays.
4. Perform initial UPS-backed power-on and verify healthy hardware POST via the `UNAS 2`'s **1.47" color LCM status screen**.

## 1.1.1 Conceptual Background: Dedicated RAID-1 Storage & the PoE++ Power Boundary

In Chapter 2, you migrated your entire Immich library, PostgreSQL database, and **WD Blue SN5000** NVMe SSD into `homelab-gpu`'s internal M.2 Slot 2. While fast and reliable over PCIe, a single NVMe drive still has **zero fault tolerance**—if its controller or NAND fails, every unbacked-up photo disappears.

The **UniFi NAS 2 (`UNAS-2`)** paired with **2× 4 TB `UACC-HDD-S-4TB`** drives solves this by introducing a dedicated, standalone **UniFi OS / UniFi Drive** storage appliance running Btrfs over RAID 1.

### 1. Hardware Deep Dive: `UNAS 2` + `2× UACC-HDD-S-4TB`

| Component | Technical Specification | Engineering Significance |
| :--- | :--- | :--- |
| **Compute & Memory (`UNAS 2`)** | Quad-core ARM Cortex-A55 @ 1.7 GHz, **4 GB RAM** | Dedicated strictly to storage I/O, Btrfs checksumming, NFS/SMB daemons, and UniFi Drive—zero contention with Docker or Immich ML! |
| **Network & Expansion** | **(1) 2.5 GbE RJ45** (2.5G/1G/100M/10M) + **(1) 5 Gbps USB-C** front port | 2.5 Gbps (~290 MB/s theoretical max) exceeds the sequential read/write ceiling of 5400 RPM HDDs (~160–200 MB/s), meaning the network never bottlenecks your disks! |
| **Drive Bays** | **(2) 3.5" SATA III (6 Gb/s)** bays *(Non-hot-swap)* | Supports RAID 1 (Basic Protection), RAID 0, or single disk. **Must be powered down before inserting or removing drives.** |
| **Hard Drives (`2× UACC-HDD-S-4TB`)** | 3.5" SATA 6 Gb/s, **5400 RPM**, **180 TB/year** workload rating, **1,000,000-hr MTBF** | 5400 RPM NAS/surveillance-class drives run cooler, quieter, and draw significantly lower spin-up and idle power than 7200 RPM enterprise drives—ideal for a PoE++-powered desktop NAS. |
| **Front Display & Setup** | **1.47" Color LCM Status Screen** + Bluetooth 4.1 | Displays IP address, storage utilization, drive bay health, and active transfer throughput at a glance. |

### 2. Why RAID 1 (~3.64 TiB Usable) Beats RAID 0 (~7.28 TiB)
With two identical 4 TB drives (`4,000,000,000,000` decimal bytes $\approx 3.638\text{ TiB}$ binary each):
- **RAID 0 (Striping — DO NOT USE for Backups):** Combines both drives into ~7.28 TiB, but **doubles your probability of total data loss**—if either drive fails, 100% of data on *both* drives is permanently destroyed.
- **RAID 1 (Mirroring — Selected Standard):** Writes every block identically to both Bay 1 and Bay 2, yielding **~3.64 TiB usable capacity** (minus a small UniFi OS/Btrfs metadata overhead). If either 4 TB HDD suffers a mechanical failure, your homelab continues running uninterrupted off the surviving disk while UniFi Drive alerts you to swap the failed drive and rebuild the mirror.
- **Remember:** *RAID is availability, not a backup!* Putting RAID 1 on the `UNAS 2` protects against HDD hardware failure, while copying data from `homelab-gpu`'s internal NVMe SSD and `homelab-pi`'s Tier-0 configs onto `UNAS 2` (and to encrypted Google Drive in Step 6) gives you true **3-2-1 backup resilience**.

### 3. CRITICAL POWER ENGINEERING: Why `USW-Enterprise-8-PoE` Cannot Directly Power `UNAS 2`

This is the single most important hardware compatibility detail in Chapter 3. Read this carefully before plugging your `UNAS 2` into your switch!

| IEEE Standard | Marketing Name | Max PSE Output (Switch) | Max PD Draw (Device) | Applies To |
| :--- | :--- | :---: | :---: | :--- |
| **IEEE 802.3af** | PoE | 15.4 W | 12.95 W | IP phones, small sensors |
| **IEEE 802.3at** | **PoE+** | **30 W** *(UI rates 32W/port)* | 25.5 W | **`USW-Enterprise-8-PoE` Ports 1–8** |
| **IEEE 802.3bt (Type 3)** | **PoE++** | **60 W** | **51 W – 60 W** | **`UNAS 2` Power Input (52W drive budget)** |

- **What happens if you plug `UNAS 2` directly into `USW-Enterprise-8-PoE`?** Two 3.5" spinning hard drives draw peak 12V surge current during simultaneous motor spin-up alongside the ARM SoC, requiring up to **60W PoE++ (`802.3bt`)**. Because the `USW-Enterprise-8-PoE` maxes out at **32W PoE+ (`802.3at`)**, direct switch power is **insufficient**—the UNAS 2 will either fail to negotiate boot power or brown out during HDD spin-up!
- **The Official Ubiquiti Solution (Included in the Box!):** Ubiquiti ships a dedicated **60W PoE++ Adapter (Injector)** right inside the `UNAS 2` box! By wiring the injector inline between `USW-Enterprise-8-PoE` **Port 4** and the `UNAS 2`:
  1. **Data** passes cleanly at full **2.5 GbE** between Switch Port 4 and the UNAS 2.
  2. **Power** (full 60W PoE++) comes directly from the injector plugged into your **UPS Battery + Surge** bank—preserving your switch's 120W PoE+ budget for future PoE+ devices (like Wi-Fi APs or a PoE-powered Raspberry Pi)!

```mermaid
flowchart LR
    UPS["UPS Battery + Surge Bank"] ==>|"120V AC"| Switch["USW-Enterprise-8-PoE<br>Port 4 (2.5 GbE | PoE+ Max 32W)"]
    UPS ==>|"120V AC"| Injector["Included 60W PoE++ Adapter<br>(Provides Full 60W 802.3bt)"]
    
    Switch ===|"0.4m Blue Patch Cable<br>(2.5 GbE Data Only -> LAN In)"| Injector
    Injector ===|"0.4m Blue Patch Cable<br>(2.5 GbE Data + 60W PoE++ -> PoE Out)"| UNAS["UniFi UNAS 2<br>2x 4TB UACC-HDD-S-4TB (RAID 1)"]
```

## 📖 Official Hardware Reference Figures

<br>

<img src="https://github.com/rd908213/Homelab-Development/blob/main/assets/hardware/unifi_unas_2.png?raw=true" width="60%">

*Reference Figure 1: Ubiquiti UniFi NAS 2 (`UNAS-2`) — 2-bay desktop NAS with 1.47" front LCM status display, front 5 Gbps USB-C port, and rear 2.5 GbE PoE++ port.*

<br>

<img src="https://github.com/rd908213/Homelab-Development/blob/main/assets/hardware/unifi_uacc_hdd_s_4tb.png?raw=true" width="45%">

*Reference Figure 2: Ubiquiti Basic Capacity 3.5" SATA HDD, 4 TB (`UACC-HDD-S-4TB`) — 5400 RPM, SATA 6 Gb/s, 180 TB/yr workload rating, 1,000,000-hr MTBF (2× installed in RAID 1).*

<br>

## 1.1.2 Step-by-Step Laboratory Guide

### Task 1.1.1: ESD-Safe Mechanical Installation of `2× UACC-HDD-S-4TB` into `UNAS 2`

> ⚠️ **CRITICAL NON-HOT-SWAP WARNING:** The `UNAS 2` drive bays are **NOT hot-swappable**. Never open the chassis or insert/remove SATA drives while the PoE++ Ethernet cable is connected. Always disconnect power completely before handling drives.

1. Unbox the **UniFi UNAS 2**, both **4 TB `UACC-HDD-S-4TB`** drives, and the included **60W PoE++ adapter** on a clean, flat, static-free surface. Touch a grounded metal chassis (or wear an ESD wrist strap) before removing the HDDs from their anti-static bags.
2. **Record Drive Serial Numbers (Pro Engineering Habit):** Before inserting the drives, snap a quick photo of both drive labels (or write down the last 6 characters of each serial number) and label which one goes into **Bay 1** vs. **Bay 2**. If a drive ever throws a SMART warning years later, you will know *immediately* which physical bay to pull without guessing!
3. Open the `UNAS 2` enclosure / drive access latch according to the included Ubiquiti Quick Start Guide and slide out the two 3.5" drive sleds/trays.
4. Mount each `UACC-HDD-S-4TB` into its 3.5" tray (aligning the side retention pins/screws firmly without flexing the drive's exposed bottom PCB). Handle each HDD strictly by its aluminum side edges—never press on the top metal lid or bottom circuit board.
5. Slide **Drive 1** and **Drive 2** straight into Bay 1 and Bay 2 until the SATA power and data connectors seat firmly and flush against the backplane, then close and secure the `UNAS 2` housing.

### 🖼️ Task 1.1.1 Verification: Dual 4TB HDD Installation & Serial Mapping

**Verification Objective:** Capture a photo of both `UACC-HDD-S-4TB` drives mounted in their trays (or seated inside the `UNAS 2` before closing the chassis), keeping track of Bay 1 vs. Bay 2.

---

> ### 📷 User Verification Screenshot Submission Box
> *Attach your milestone verification photo below (drag & drop an image or use a Markdown image link):*
> 
> ```markdown
> <!-- Insert your screenshot here: -->
> 
> ```

### Task 1.1.2: Wire the UPS-Backed 60W PoE++ Injector & Verify Initial Boot
1. Plug the **included 60W PoE++ Adapter's** AC power cord into a **Battery + Surge** outlet on your external UPS (right alongside your `USW-Enterprise-8-PoE`, `homelab-pi`, and `homelab-gpu`).
   - *Why UPS protection is critical for a RAID 1 HDD NAS:* Sudden AC power loss mid-write while spinning platters are active can cause RAID mirror desynchronization ("write hole") or Btrfs journal replay delays.
2. Inspect the two RJ45 ports on the 60W PoE++ Injector carefully—do **not** swap them!
   - **`LAN` (Data In):** Connect a **`0.4m` UniFi Patch Cable** (`UACC-CABLE-PATCH-0.4M-BL-10`) from **`USW-Enterprise-8-PoE` Port 4** $\rightarrow$ Injector **`LAN`** port.
   - **`PoE` (Data + 60W PoE++ Out):** Connect a second **`0.4m` UniFi Patch Cable** from Injector **`PoE`** port $\rightarrow$ **`UNAS 2`** rear 2.5 GbE RJ45 port.
3. Observe the **1.47" front LCM screen** on the `UNAS 2`:
   - Within ~10 seconds, the LED ring illuminates and the LCM displays the Ubiquiti boot animation.
   - Listen briefly for the gentle, steady spin-up hum of both 5400 RPM drives (no loud rhythmic clicking).
   - After ~60–90 seconds, the LCM will show **"Ready for Setup"** (or display its initial DHCP IP address on your `10.0.0.x` subnet).
4. **Do NOT initialize the storage pool yet!** First, in Module 1.2, we will complete structured cabling for the entire shelf and bind static DHCP IP reservations (`10.0.0.202` for `UNAS 2` and `10.0.0.203` for `USW-Enterprise-8-PoE`) so every device boots onto its permanent address before software onboarding.

### 🖼️ Task 1.1.2 Verification: `UNAS 2` Powered via 60W PoE++ Injector & LCM Ready

**Verification Objective:** Capture a photo showing the `UNAS 2` powered on via its inline 60W PoE++ adapter with its 1.47" front LCM display illuminated and ready for setup.

---

> ### 📷 User Verification Screenshot Submission Box
> *Attach your milestone verification photo below (drag & drop an image or use a Markdown image link):*
> 
> ```markdown
> <!-- Insert your screenshot here: -->
> 
> ```

## 🏁 Module Checkpoint & Reflection

Before proceeding to the next guide, confirm the following checklist:

- [ ] Both `UACC-HDD-S-4TB` drives installed into `UNAS 2` Bay 1 & Bay 2 while fully powered off, with serial numbers recorded.
- [ ] Understood why `USW-Enterprise-8-PoE` (PoE+, 32W) cannot directly power `UNAS 2` (PoE++, 60W).
- [ ] Included 60W PoE++ injector plugged into the UPS **Battery + Surge** bank and wired correctly (`Port 4 -> LAN In`, `PoE Out -> UNAS 2`).
- [ ] `UNAS 2` powered on cleanly with both drives spinning smoothly and the 1.47" LCM screen active.
- [ ] Verification photos captured and attached above.

Proceed to [Module 1.2: Structured 0.4m Patch Cabling, Multi-Gig Bottleneck Analysis & Static IP Reservations](./01_2_structured_patch_cabling_and_25gbe_topology.ipynb).